# Basics &mdash; Complement, Relative to a Universe

**Concept 4 of the Basics decomposition:** *Complement, Relative to a Universe*

$\overline{S} = U - S$ &mdash; meaningless until the universe $U$ is fixed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Complement-Relative-To-Universe/Concept-Complement-Relative-To-Universe.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


$$\overline{S} = U - S$$

The complement is **relative to a universe** $U$, and the notation hides it. Until $U$
is fixed, $\overline{S}$ means nothing.

For languages the universe is $\Sigma^*$, so $\overline{L} = \Sigma^* - L$ &mdash; and
$\Sigma$ matters. The complement of $\{0^n\}$ over $\{0\}$ is *not* the complement of
$\{0^n\}$ over $\{0,1\}$.

This is the same point Chapter 6 makes about DFA complementation: **totalize first**.
Totalizing is exactly the act of making the universe explicit, so that every string in
$\Sigma^*$ has a verdict and flipping $F$ means what you want.

Jove's `lcomplem(L, sigma, n)` takes the alphabet and a length bound, because an
infinite complement cannot be listed.

## 2. Definitions

### Complement needs a universe

In [ ]:
S = {1, 2, 3}
for name, U in [('0..5', set(range(6))), ('0..9', set(range(10))),
                ('the odds under 10', {1, 3, 5, 7, 9})]:
    print("  U = %-18s  complement of {1,2,3} = %s" % (name, sorted(U - S)))

### For languages, the universe is $\Sigma^*$

In [ ]:
from itertools import product
def sigma_star(sigma, n):
    return {''.join(p) for k in range(n + 1) for p in product(sorted(sigma), repeat=k)}

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;3.&nbsp;Powerset](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Powerset/Concept-Powerset.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;5.&nbsp;Equivalence Relations and Partitioning](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Equivalence-Relations/Concept-Equivalence-Relations.ipynb)&nbsp;&rarr;

---

## 3. Tests

The same language has different complements over different alphabets.

In [ ]:
L = {'', '0', '00', '000'}                   # 0^n, n <= 3
c1 = sigma_star({'0'}, 3) - L
c2 = sigma_star({'0', '1'}, 3) - L
print("over {0}   : complement =", sorted(c1))
print("over {0,1} : complement =", sorted(c2)[:8], "... (%d strings)" % len(c2))
assert c1 == set() and len(c2) > 0
print("\nOver {0} the complement is EMPTY; over {0,1} it is almost everything.")

Jove's `lcomplem` makes the alphabet and the bound explicit.

In [ ]:
L = {'', '0', '00'}
for sig in [{'0'}, {'0', '1'}]:
    got = lcomplem(L, sig, 3)
    print("  lcomplem(L, %-8s, 3) -> %s" % (str(sorted(sig)), sorted(got)[:8]))
    assert got == sigma_star(sig, 3) - L
print("\nagrees with Sigma* - L in both cases")

Double complement returns the original.

In [ ]:
for sig in [{'0'}, {'0', '1'}]:
    once  = lcomplem(L, sig, 3)
    twice = lcomplem(once, sig, 3)
    print("  over %-8s : complement twice == L ? %s" % (str(sorted(sig)), twice == L))
    assert twice == L

**The DFA version of the same point:** totalize before flipping.

In [ ]:
partial = md2mc('''DFA
IF : 0 -> Od
Od : 0 -> IF
''')
print("Sigma of the partial machine :", sorted(partial["Sigma"]))
wide  = addtosigma_dfa(partial, {'1'})
right = comp_dfa(wide)               # totalizes, THEN flips
tot   = totalize_dfa(wide)
strs = sorted(sigma_star({'0', '1'}, 4))
assert all(accepts_dfa(right, s) == (not accepts_dfa(tot, s)) for s in strs)
print("complement correct on all %d strings up to length 4" % len(strs))
print()
print("Totalizing IS fixing the universe: it gives every string in Sigma* a")
print("verdict, so 'the ones not accepted' is a well-defined set.")

## 4. Animation

The complemented machine &mdash; same shape, opposite double circles.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(comp_dfa(md2mc('DFA\nIF : 0 -> Od\nIF : 1 -> IF\nOd : 0 -> IF\nOd : 1 -> Od\n')), FuseEdges=True)

## 5. Exercises


1. What is $\overline{\Sigma^*}$? What is $\overline{\emptyset}$?
2. Why does `lcomplem` need a length bound at all?
3. Give two sets with the same complement under different universes.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Complement-Relative-To-Universe')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')